## Warping Images into Alignment

# Warping and Canvas Alignment with Homographies

Welcome back! In Units 1 and 2, you learned how to find keypoints across two images, match them, and use the RANSAC algorithm to compute an accurate homography matrix.

In Unit 3, we put that homography matrix to practical use. A homography becomes visible when applied to warp pixels. To execute this, we assign distinct roles to our two images:

* **Source image:** The image being transformed, stretched, and moved (typically the left image).
* **Target image:** The image that remains fixed in its reference coordinate frame (typically the right image).

> **Mental Model:** Think of the source image as a photo printed on a flexible rubber sheet. The homography matrix is the mathematical transformation defining how to pull, stretch, and align that sheet so its landmarks coincide with the target photo.

---

## Finding the New Canvas Boundaries

Transforming the source image displaces its pixels to new coordinates. In many cases, pixels are mapped to **negative coordinates** (extending above or to the left of the origin) or beyond the right/bottom borders of the default frame. Without resizing the canvas, the warped output gets clipped.

To avoid clipping, we determine where the four corners of the source image will land after transformation:

```python
import cv2
import numpy as np

# Assume 'source' and 'target' are already loaded images
h1, w1 = source.shape[:2]
h2, w2 = target.shape[:2]

# Define the 4 corners of the source image [x, y]
source_corners = np.float32(
    [[0, 0], [w1, 0], [w1, h1], [0, h1]]
).reshape(-1, 1, 2)

# Define the 4 corners of the target image [x, y]
target_corners = np.float32(
    [[0, 0], [w2, 0], [w2, h2], [0, h2]]
).reshape(-1, 1, 2)

```

OpenCV geometric functions expect 2D points in a 3D array shape of `(N, 1, 2)`. We use `.reshape(-1, 1, 2)` to match this convention.

Next, apply the homography transformation to `source_corners` using `cv2.perspectiveTransform`, which operates directly on point coordinates rather than raster images:

```python
# Transform source corners via the homography matrix
warped_corners = cv2.perspectiveTransform(source_corners, homography)

# Combine warped source corners with fixed target corners
all_corners = np.concatenate([warped_corners, target_corners], axis=0)
print(all_corners.shape)

```

```text
Output: (8, 1, 2)

```

With all eight corner locations aggregated, compute the minimum and maximum $x$ and $y$ bounds spanning both images:

```python
# Determine bounding limits
xmin, ymin = np.floor(all_corners.min(axis=0).ravel()).astype(int)
xmax, ymax = np.ceil(all_corners.max(axis=0).ravel()).astype(int)

print(f"Canvas boundaries: xmin={xmin}, ymin={ymin}, xmax={xmax}, ymax={ymax}")

```

```text
Output: Canvas boundaries: xmin=-250, ymin=-50, xmax=1000, ymax=600

```

Because the source image shifted left and up to align with the target, `xmin` and `ymin` evaluate to negative values.

---

## Shifting with a Translation Matrix

Standard image arrays cannot render negative pixel indices ($x < 0$ or $y < 0$). To prevent clipping, the origin must be offset by shifting all coordinates right and down:

$$\text{shift\_x} = -x_{\text{min}}, \quad \text{shift\_y} = -y_{\text{min}}$$

```python
shift_x = -xmin
shift_y = -ymin
print(f"Shift X: {shift_x}, Shift Y: {shift_y}")

```

```text
Output: Shift X: 250, Shift Y: 50

```

To incorporate this offset into OpenCV transformations, construct a $3 \times 3$ affine translation matrix:

```python
translation = np.array(
    [
        [1, 0, shift_x],
        [0, 1, shift_y],
        [0, 0, 1],
    ],
    dtype=np.float64,
)

```

The diagonal values preserve scale, while `shift_x` and `shift_y` translate the pixel coordinate frame into positive index space.

---

## Applying the Warp

Calculate the dimensions for the unified canvas:

```python
canvas_width = xmax - xmin
canvas_height = ymax - ymin
size = (canvas_width, canvas_height)

```

Warp the source image into the expanded canvas using `cv2.warpPerspective`. To translate and warp in a single pass, compose the transformation matrices using matrix multiplication (`@`):

$$\mathbf{H}_{\text{composite}} = \mathbf{T} \cdot \mathbf{H}$$

```python
# Apply translation and homography in a single operation
warped_source = cv2.warpPerspective(source, translation @ homography, size)

```

`warped_source` contains the source image warped into alignment and positioned within valid positive coordinates.

---

## Bringing the Images Together

To verify alignment, place the fixed target image onto the same expanded coordinate plane:

1. Create a blank black canvas matching `warped_source` in size and data type.
2. Insert the target image with the matching translation offset applied via NumPy slicing:

```python
# Create an empty canvas
target_canvas = np.zeros_like(warped_source)

# Place the target image at the offset position
target_canvas[shift_y : shift_y + h2, shift_x : shift_x + w2] = target

```

Blend both layers using `cv2.addWeighted` to inspect registration accuracy:

```python
# 50/50 alpha blend for visual verification
preview = cv2.addWeighted(warped_source, 0.5, target_canvas, 0.5, 0)

```

If the homography and coordinate shifts are accurate, overlapping structural details (such as edges and corners) will register cleanly.

---

## Summary & Key Steps

| Step | Operation | Key OpenCV / NumPy Function |
| --- | --- | --- |
| **1. Boundary Detection** | Project corner coordinates through homography | `cv2.perspectiveTransform` |
| **2. Offset Calculation** | Compute coordinate extrema ($x_{\min}, y_{\min}$) and build offset | `np.min()`, `np.max()`, `3x3 Translation Matrix` |
| **3. Coordinate Warping** | Compose matrices and warp source image | `cv2.warpPerspective(src, T @ H, size)` |
| **4. Canvas Blending** | Offset target image and overlay layers | NumPy Slicing & `cv2.addWeighted` |

## Mapping Corners Across Two Images

Nice work obtaining a solid homography from RANSAC in the previous unit — now it is time to put that matrix to use and begin warping images into a shared canvas.

In this first step of the unit, you will begin completing the warp_to_canvas function inside geometry.py. The objective is to determine where the source image lands after the warp and to track the target image's corners as well.

Open geometry.py and follow the TODO comments inside warp_to_canvas:

    Read the heights and widths of source and target into (h1, w1) and (h2, w2).
    Build source_corners and target_corners as float32 arrays in the order [0, 0], [w, 0], [w, h], [0, h], reshaped to (-1, 1, 2).
    Use cv2.perspectiveTransform to map source_corners through the homography into warped_corners.
    Concatenate warped_corners and target_corners along axis=0 into all_corners.

Once this component is in place, we will have everything required to compute the bounding box of the full canvas in the next exercise.

```
import cv2
import numpy as np


def matched_points(kp1, kp2, matches):
    src = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    dst = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)
    return src, dst


def estimate_homography(kp1, kp2, matches, ransac_threshold=5.0):
    if len(matches) < 4:
        raise ValueError("At least four matches are required")

    src, dst = matched_points(kp1, kp2, matches)
    homography, mask = cv2.findHomography(src, dst, cv2.RANSAC, ransac_threshold)

    if homography is None or mask is None:
        raise ValueError("Homography estimation failed")

    return homography, mask.ravel().astype(bool)


def warp_to_canvas(source, target, homography):
    """
    Warp source into target's coordinate frame.

    The homography must map source points to target points.
    In the course scripts, source is usually the left image and target is
    usually the right image.
    """
    # We will build this function step by step across this unit.
    # In this exercise, we only set up the corners and combine them
    # into all_corners.

    # TODO: Read source and target shapes into (h1, w1) and (h2, w2).

    # TODO: Build source_corners as a float32 array of the four source
    # corners in this order: [0, 0], [w1, 0], [w1, h1], [0, h1].
    # Then reshape it to (-1, 1, 2).

    # TODO: Build target_corners the same way for the target image,
    # using w2 and h2.

    # TODO: Call cv2.perspectiveTransform on source_corners with the
    # homography to get warped_corners.

    # TODO: Concatenate warped_corners and target_corners along axis=0
    # into a variable named all_corners.

```

Here is the completed implementation for `geometry.py`:

```python
import cv2
import numpy as np


def matched_points(kp1, kp2, matches):
    src = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    dst = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)
    return src, dst


def estimate_homography(kp1, kp2, matches, ransac_threshold=5.0):
    if len(matches) < 4:
        raise ValueError("At least four matches are required")

    src, dst = matched_points(kp1, kp2, matches)
    homography, mask = cv2.findHomography(src, dst, cv2.RANSAC, ransac_threshold)

    if homography is None or mask is None:
        raise ValueError("Homography estimation failed")

    return homography, mask.ravel().astype(bool)


def warp_to_canvas(source, target, homography):
    """
    Warp source into target's coordinate frame.

    The homography must map source points to target points.
    In the course scripts, source is usually the left image and target is
    usually the right image.
    """
    # 1. Read source and target shapes
    h1, w1 = source.shape[:2]
    h2, w2 = target.shape[:2]

    # 2. Build corner coordinate arrays
    source_corners = np.float32([[0, 0], [w1, 0], [w1, h1], [0, h1]]).reshape(-1, 1, 2)
    target_corners = np.float32([[0, 0], [w2, 0], [w2, h2], [0, h2]]).reshape(-1, 1, 2)

    # 3. Transform source corners using the homography
    warped_corners = cv2.perspectiveTransform(source_corners, homography)

    # 4. Concatenate transformed source corners with target corners
    all_corners = np.concatenate([warped_corners, target_corners], axis=0)

```

## Building the Full Warp Pipeline

Nice work mapping the corners through the homography in the previous step. Now, it is time to turn those corner coordinates into a real canvas and warp the source onto it.

Open geometry.py and follow the TODO comments inside warp_to_canvas. You will work below the line where all_corners is built; since each step depends on the one before it, it helps to tackle them in order.

Here is the plan:

    Find xmin, ymin, xmax, and ymax from all_corners using np.floor and np.ceil, then cast them to int.
    Compute shift_x and shift_y as the negatives of xmin and ymin so that no coordinates are negative.
    Build a 3x3 np.float64 translation matrix that places shift_x and shift_y in the third column.
    Compute the canvas size as (xmax - xmin, ymax - ymin) (the validation check immediately following expects this exact name).

Once the setup is ready, finish the function:

    Warp the source with cv2.warpPerspective using translation @ homography and the canvas size, then store the result in warped_source.
    Create target_canvas as a black image with the same shape as warped_source using np.zeros_like.
    Paste the target into target_canvas using the slice [shift_y : shift_y + h2, shift_x : shift_x + w2].
    Return warped_source and target_canvas as a tuple.

Get this working, and you will have a function that can warp any image pair onto a clean shared canvas — the key piece you will reuse when stitching panoramas next.

```
import cv2
import numpy as np


def matched_points(kp1, kp2, matches):
    src = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    dst = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)
    return src, dst


def estimate_homography(kp1, kp2, matches, ransac_threshold=5.0):
    if len(matches) < 4:
        raise ValueError("At least four matches are required")

    src, dst = matched_points(kp1, kp2, matches)
    homography, mask = cv2.findHomography(src, dst, cv2.RANSAC, ransac_threshold)

    if homography is None or mask is None:
        raise ValueError("Homography estimation failed")

    return homography, mask.ravel().astype(bool)


def warp_to_canvas(source, target, homography):
    """
    Warp source into target's coordinate frame.

    The homography must map source points to target points.
    In the course scripts, source is usually the left image and target is
    usually the right image.
    """
    h1, w1 = source.shape[:2]
    h2, w2 = target.shape[:2]

    source_corners = np.float32(
        [[0, 0], [w1, 0], [w1, h1], [0, h1]]
    ).reshape(-1, 1, 2)

    target_corners = np.float32(
        [[0, 0], [w2, 0], [w2, h2], [0, h2]]
    ).reshape(-1, 1, 2)

    warped_corners = cv2.perspectiveTransform(source_corners, homography)
    all_corners = np.concatenate([warped_corners, target_corners], axis=0)

    # TODO: Find the canvas boundaries. Compute xmin, ymin from
    # all_corners.min(axis=0).ravel() using np.floor, and xmax, ymax from
    # all_corners.max(axis=0).ravel() using np.ceil. Cast both to int with
    # .astype(int).

    # TODO: Compute how far we need to push pixels into the positive
    # region. Set shift_x = -xmin and shift_y = -ymin.

    # TODO: Build a 3x3 translation matrix as a np.float64 array using
    # shift_x and shift_y on the third column. The diagonal should be 1s.

    # TODO: Compute the final canvas size as (xmax - xmin, ymax - ymin).
    # The lines below validate the canvas size; leave them as is.
    # if size[0] <= 0 or size[1] <= 0:
    #     raise ValueError("Invalid warp canvas size")

    # TODO: Use cv2.warpPerspective to warp the source. The combined
    # matrix is translation @ homography, and the output size is the
    # tuple you just computed. Assign the result to warped_source.

    # TODO: Create target_canvas as a black image with the same shape and
    # dtype as warped_source using np.zeros_like.

    # TODO: Place the target image inside target_canvas using slicing.
    # The slice should be [shift_y : shift_y + h2, shift_x : shift_x + w2].

    # TODO: Return warped_source and target_canvas as a tuple.

```

Here is the completed `geometry.py` file with the full `warp_to_canvas` pipeline implemented:

```python
import cv2
import numpy as np


def matched_points(kp1, kp2, matches):
    src = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    dst = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)
    return src, dst


def estimate_homography(kp1, kp2, matches, ransac_threshold=5.0):
    if len(matches) < 4:
        raise ValueError("At least four matches are required")

    src, dst = matched_points(kp1, kp2, matches)
    homography, mask = cv2.findHomography(src, dst, cv2.RANSAC, ransac_threshold)

    if homography is None or mask is None:
        raise ValueError("Homography estimation failed")

    return homography, mask.ravel().astype(bool)


def warp_to_canvas(source, target, homography):
    """
    Warp source into target's coordinate frame.

    The homography must map source points to target points.
    In the course scripts, source is usually the left image and target is
    usually the right image.
    """
    h1, w1 = source.shape[:2]
    h2, w2 = target.shape[:2]

    source_corners = np.float32(
        [[0, 0], [w1, 0], [w1, h1], [0, h1]]
    ).reshape(-1, 1, 2)

    target_corners = np.float32(
        [[0, 0], [w2, 0], [w2, h2], [0, h2]]
    ).reshape(-1, 1, 2)

    warped_corners = cv2.perspectiveTransform(source_corners, homography)
    all_corners = np.concatenate([warped_corners, target_corners], axis=0)

    # 1. Compute canvas bounding box limits
    xmin, ymin = np.floor(all_corners.min(axis=0).ravel()).astype(int)
    xmax, ymax = np.ceil(all_corners.max(axis=0).ravel()).astype(int)

    # 2. Calculate coordinate offsets
    shift_x = -xmin
    shift_y = -ymin

    # 3. Build 3x3 affine translation matrix
    translation = np.array(
        [
            [1.0, 0.0, shift_x],
            [0.0, 1.0, shift_y],
            [0.0, 0.0, 1.0],
        ],
        dtype=np.float64,
    )

    # 4. Canvas size tuple (width, height)
    size = (xmax - xmin, ymax - ymin)
    if size[0] <= 0 or size[1] <= 0:
        raise ValueError("Invalid warp canvas size")

    # 5. Warp source image into expanded coordinate plane
    warped_source = cv2.warpPerspective(source, translation @ homography, size)

    # 6. Initialize black target canvas
    target_canvas = np.zeros_like(warped_source)

    # 7. Paste target image at shifted position
    target_canvas[shift_y : shift_y + h2, shift_x : shift_x + w2] = target

    # 8. Return warped layers
    return warped_source, target_canvas

```

## Previewing the Aligned Image Blend

Nice work finishing the warp_to_canvas pipeline in the previous exercise! Now, it is time to plug that function into the main script so you can actually see the alignment on screen.

Open solution.py and follow the TODO comments to wire everything together.

Your job is to:

    Import cv2 at the top of the file and update the geometry import to also bring in warp_to_canvas.
    Add a new --out CLI argument with a default of "warped.jpg" so the preview image has a place to be saved.
    Call warp_to_canvas(left, right, homography) and unpack the result into warped and target_canvas.
    Blend the two images at 50/50 using cv2.addWeighted(warped, 0.5, target_canvas, 0.5, 0) and store it in a variable named preview.
    Save preview with cv2.imwrite(args.out, preview), then show it with cv2.imshow("alignment preview", preview), followed by cv2.waitKey(0) and cv2.destroyAllWindows().

Run the program in the terminal, for example: python3 solution.py sample_images/building/1.jpg sample_images/building/2.jpg sample_images/building/3.jpg

Once everything is connected, you will have a working tool that turns two photos and a homography into a blended preview image — a big step toward stitching full panoramas!

```
import argparse

# TODO: Import cv2 so you can blend, save, and display the preview image.

from cvkit import preprocess_for_features, read_color
from features import detect_and_compute, match_descriptors
# TODO: Update this import to also bring in warp_to_canvas from geometry.
from geometry import estimate_homography


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("left")
    parser.add_argument("right")
    parser.add_argument("--method", choices=["sift", "orb", "akaze"], default="sift")
    parser.add_argument("--ratio", type=float, default=0.75)
    parser.add_argument("--ransac-threshold", type=float, default=5.0)
    # TODO: Add a new --out CLI argument with a default of "warped.jpg".
    # This is where the blended preview image will be saved.
    args = parser.parse_args()

    left = read_color(args.left)
    right = read_color(args.right)

    kp1, des1 = detect_and_compute(
        preprocess_for_features(left),
        method=args.method,
    )
    kp2, des2 = detect_and_compute(
        preprocess_for_features(right),
        method=args.method,
    )

    matches = match_descriptors(des1, des2, ratio=args.ratio)
    homography, inliers = estimate_homography(
        kp1,
        kp2,
        matches,
        ransac_threshold=args.ransac_threshold,
    )

    # TODO: Call warp_to_canvas(left, right, homography) and unpack the
    # result into two variables: warped and target_canvas.

    # TODO: Blend warped (weight 0.5) and target_canvas (weight 0.5) with
    # cv2.addWeighted and store the result in a variable named preview.

    print("matches:", len(matches))
    print("inliers:", int(inliers.sum()))
    print("inlier ratio:", float(inliers.mean()))

    # TODO: Save the preview image to disk using cv2.imwrite(args.out, preview).

    # TODO: Show the preview in a window using cv2.imshow("alignment preview", preview),
    # then call cv2.waitKey(0) and cv2.destroyAllWindows().


if __name__ == "__main__":
    main()

```

Here is the completed `solution.py` file with all imports, CLI arguments, warping, blending, and previewing logic implemented:

```python
import argparse
import cv2

from cvkit import preprocess_for_features, read_color
from features import detect_and_compute, match_descriptors
from geometry import estimate_homography, warp_to_canvas


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("left")
    parser.add_argument("right")
    parser.add_argument("--method", choices=["sift", "orb", "akaze"], default="sift")
    parser.add_argument("--ratio", type=float, default=0.75)
    parser.add_argument("--ransac-threshold", type=float, default=5.0)
    parser.add_argument("--out", default="warped.jpg")
    args = parser.parse_args()

    left = read_color(args.left)
    right = read_color(args.right)

    kp1, des1 = detect_and_compute(
        preprocess_for_features(left),
        method=args.method,
    )
    kp2, des2 = detect_and_compute(
        preprocess_for_features(right),
        method=args.method,
    )

    matches = match_descriptors(des1, des2, ratio=args.ratio)
    homography, inliers = estimate_homography(
        kp1,
        kp2,
        matches,
        ransac_threshold=args.ransac_threshold,
    )

    # 1. Warp source (left) and shift target (right) onto the shared canvas
    warped, target_canvas = warp_to_canvas(left, right, homography)

    # 2. Blend the warped source and target canvas 50/50
    preview = cv2.addWeighted(warped, 0.5, target_canvas, 0.5, 0)

    print("matches:", len(matches))
    print("inliers:", int(inliers.sum()))
    print("inlier ratio:", float(inliers.mean()))

    # 3. Save preview image to disk
    cv2.imwrite(args.out, preview)

    # 4. Show preview window and wait for keypress
    cv2.imshow("alignment preview", preview)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

```